In [0]:
from pyspark.sql.functions import col, lower, to_date

# 1. Define the path to your Volume
# Format: /Volumes/<catalog>/<schema>/<volume>/<file_name>
volume_path = "/Volumes/workspace/krishigati/data_landing/Agriculture_price_dataset.csv"

# 2. Read the raw CSV
# We use header=True because your CSV has column names like 'District Name'
df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(volume_path)

# 3. Rename columns to remove spaces (required for Delta tables)
df = df.withColumnRenamed("District Name", "District_Name") \
       .withColumnRenamed("Market Name", "Market_Name") \
       .withColumnRenamed("Price Date", "Price_Date")

# 4. Filter for Nashik District and remove Market_Name containing 'ghoti'
# We use lower() to ensure we catch all variations (Nashik, nashik, NASHIK)
cleaned_df = df.filter(
    (lower(col("District_Name")) == "nashik") &
    (~lower(col("Market_Name")).contains("ghoti"))
)

# 4.1. Keep only rows where Commodity is onion or wheat
cleaned_df = cleaned_df.filter(
    lower(col("Commodity")).isin("onion", "wheat")
)

# 5. Standardize the Date format
# This is required for Member 2 to perform the 3-day forecasting
cleaned_df = cleaned_df.withColumn("Standard_Date", to_date(col("Price_Date"), "M/d/yyyy"))

# 6. Save as a Delta Table (Silver Layer)
# We will save this in the 'krishigati' schema for easy access
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.krishigati")
cleaned_df.write.format("delta").mode("overwrite").saveAsTable("workspace.krishigati.nashik_mandi_silver")

print(f"Success! Cleaned {cleaned_df.count()} rows.")
print("The data is now available in the table: workspace.krishigati.nashik_mandi_silver")

Success! Cleaned 9013 rows.
The data is now available in the table: workspace.krishigati.nashik_mandi_silver


In [0]:
mandi_coords = {
    "Lasalgaon": (20.1437, 74.2231),
    "Lasalgaon(Niphad)": (20.1437, 74.2231),
    "Lasalgaon(Vinchur)": (20.1171, 74.2305),
    "Pimpalgaon": (20.1656, 73.9856),
    "Pimpalgaon Baswant(Saykheda)": (20.1100, 74.0500),
    "Nasik": (20.0050, 73.7889),
    "Malegaon": (20.5524, 74.5244),
    "Nandgaon": (20.3117, 74.6548),
    "Satana": (20.5900, 74.2000),
    "Kalvan": (20.4851, 73.9169),
    "Manmad": (20.2483, 74.4379),
    "Dindori": (20.2012, 73.8322),
    "Dindori(Vani)": (20.3200, 73.8900),
    "Devala": (20.3500, 74.1500),
    "Chandvad": (20.3306, 74.2464),
    "Sinner": (19.8486, 74.0019),
    "Yeola": (20.0353, 74.4856),
    "Nampur": (20.5843, 74.1167),
    "Umrane": (20.4000, 74.2800),
    "Suragana": (20.4100, 73.6100),
    "Shree Rameshwar Krushi Market": (20.2500, 74.1000),
    "Premium Krushi Utpanna Bazar": (20.0000, 73.8000),
    "Malharshree Farmers Producer Co Ltd": (20.3000, 74.2000),
    "Mankamneshwar Farmar Producer CoLtd Sanchalit Mank": (20.1500, 74.0000),
    "Shivsiddha Govind Producer Company Limited Sanchal": (20.1000, 73.9000)
}

# Create a Pandas DataFrame from the dictionary
import pandas as pd

mandi_df = pd.DataFrame([
    {"District_Name": "Nashik", "Market_Name": k, "latitude": v[0], "longitude": v[1]}
    for k, v in mandi_coords.items()
])

# Convert to Spark DataFrame
mandi_spark_df = spark.createDataFrame(mandi_df)

# Save as Delta Table
mandi_spark_df.write.format("delta").mode("overwrite").saveAsTable("workspace.krishigati.nashik_mandi_coords")

display(mandi_spark_df)

District_Name,Market_Name,latitude,longitude
Nashik,Lasalgaon,20.1437,74.2231
Nashik,Lasalgaon(Niphad),20.1437,74.2231
Nashik,Lasalgaon(Vinchur),20.1171,74.2305
Nashik,Pimpalgaon,20.1656,73.9856
Nashik,Pimpalgaon Baswant(Saykheda),20.11,74.05
Nashik,Nasik,20.005,73.7889
Nashik,Malegaon,20.5524,74.5244
Nashik,Nandgaon,20.3117,74.6548
Nashik,Satana,20.59,74.2
Nashik,Kalvan,20.4851,73.9169


In [0]:
from pyspark.sql.functions import collect_set

# Read the silver table
silver_df = spark.table("workspace.krishigati.nashik_mandi_silver")

# Group by Market_Name and collect unique commodities
market_commodities_df = silver_df.groupBy("Market_Name").agg(collect_set("Commodity").alias("Commodities_Sold"))

# Save as Delta Table in workspace.krishigati schema
market_commodities_df.write.format("delta").mode("overwrite").saveAsTable("workspace.krishigati.nashik_market_commodities")

display(market_commodities_df)

Market_Name,Commodities_Sold
Lasalgaon(Niphad),"List(Wheat, Onion)"
Lasalgaon(Vinchur),"List(Wheat, Onion)"
Pimpalgaon,List(Onion)
Malegaon,List(Wheat)
Nandgaon,"List(Wheat, Onion)"
Pimpalgaon Baswant(Saykheda),List(Onion)
Satana,"List(Onion, Wheat)"
Lasalgaon,"List(Onion, Wheat)"
Kalvan,"List(Onion, Wheat)"
Manmad,"List(Onion, Wheat)"
